<a href="https://colab.research.google.com/github/JanTeichertKluge/demand-analysis-labs/blob/main/v2_ext/02_finetune_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Causal Fine-Tuning of Multimodal Product Embeddings

**Note:** This notebook requires a GPU. In Colab, select *Runtime → Change runtime type → T4 GPU* (or better). This version trains on a fine-tuning sample of the paper's size, about four times as many rows as the subsample version. On a T4, one epoch takes roughly half an hour, so the full run needs several hours and usually more than one Colab session; an L4 or A100 is considerably faster. Whenever a session ends, run the notebook again: training resumes after the last completed epoch.

This notebook is the second of five labs that accompany [Bach et al. (2025)](https://arxiv.org/abs/2501.00382). In Lab 1, a regression of quantity on price suggested that demand is almost perfectly inelastic, which is not plausible. The missing controls are product quality and visibility. Neither is observed as a number, but both are reflected in the product description and the product image.

In this lab, we map the texts and the images into numeric vectors, so-called *embeddings*. We train a multimodal neural network that combines RoBERTa for the text, BEiT for the image and SAINT for the tabular variables, and we save the resulting embeddings for Labs 3, 4 and 5.

## Why Not Use Pre-Trained Embeddings?

A pre-trained encoder produces a representation that is optimized for the task it was trained on, such as predicting masked words or classifying images. Nothing about that task makes the representation particularly useful for predicting prices and quantities, which is what we need.

Therefore, we fine-tune the encoders. Following the paper, the network has two prediction heads, one for the quantity signal and one for the price signal, and minimizes

$$
L = \bigl\|Q_{it} - \hat{\ell}(X^{in}_i)\bigr\|_{\text{pred},2} \cdot
      \bigl\|P_{it} - \hat{m}(X^{in}_i)\bigr\|_{\text{pred},2},
$$

the product of the two root-mean-squared prediction errors. The two heads estimate the same nuisance functions that the partialling-out estimator in Lab 4 requires, $\ell(S) := E[Q \mid S]$ and $m(S) := E[P \mid S]$. The paper refers to training the representation on this objective as **causal fine-tuning**: the embedding is shaped by the estimation problem in which it will be used rather than by a generic pre-training task.

## Architecture

$$
X^{in}_i = \begin{pmatrix}\text{Text}_i \\ \text{Image}_i \\ \text{Tabular}_i\end{pmatrix}
\;\xrightarrow{\;e\;}\;
E_i = \begin{pmatrix}T_i \\ I_i \\ \mathrm{Tab}_i\end{pmatrix}
\;\xrightarrow{\;m\;}\;
\{\hat{Q}_{it}, \hat{P}_{it}\}
$$

Each encoder produces a dense vector. The three vectors are layer-normalized and concatenated into $E_i \in \mathbb{R}^{2048}$: 768 dimensions for the text, 768 for the image, and 32 for each of the 16 tabular columns. Two small multilayer perceptrons take this embedding as input and predict the two signals. Since the gradients flow back into the transformers, the embedding itself adapts during training.

The network is trained on the fine-tuning products only. Labs 4 and 5 estimate elasticities on the estimation products, whose embeddings are computed by the network but which are never used for training. Addendum A of the paper relies on this independence to show that estimating the embeddings does not affect the validity of DML inference.

## Time-Varying Inputs

The description and the image do not change over time, but part of the tabular input does: the offer counts, the lightning-deal indicator and the Buy Box fulfillment indicator are observed in every period. Following the paper's pipeline, we therefore compute an embedding for every product-period. The text and image parts, 1,536 of the 2,048 dimensions, are identical across the periods of a product, so most of the embedding cannot explain *changes* in prices. Lab 3 quantifies this, and Labs 4 and 5 build on it.

## Installation

First, we install the required packages. The installation needs a short explanation. `doubleml-deep` imports its SAINT tabular model when the package is imported, so `pytorch-widedeep` has to be importable even though we barely use it. A regular installation of `pytorch-widedeep` pulls in `spacy`, `gensim` and `opencv` and pins `numpy<2`, which downgrades numpy underneath Colab's pre-installed torch, pandas and scipy and breaks the runtime.

Instead, we install it with `--no-deps` and provide two small stub modules for the text utilities that `pytorch-widedeep` imports but that we never call. All other dependencies (`torchvision`, `cv2`, `wrapt`) are already available on Colab.

In [ ]:
%pip install -q lightning torchmetrics einops doubleml
%pip install -q --no-deps pytorch-widedeep
%pip install -q git+https://github.com/DoubleML/doubleml-deep.git

The following cell creates the two stub modules and adds them to the Python path.

In [ ]:
import os
import sys
import pathlib

# --- shim: satisfy pytorch-widedeep's unused text-utility imports -----------
# pytorch_widedeep.utils.text_utils imports gensim and spacy at module level.
# We never tokenise text with them (RoBERTa does that), so tiny stubs suffice
# and we avoid the numpy<2 downgrade their real versions would force.
SHIM = "/content/_wd_shim" if os.path.isdir("/content") else "_wd_shim"
for pkg in ("gensim", "spacy"):
    pathlib.Path(f"{SHIM}/{pkg}").mkdir(parents=True, exist_ok=True)

pathlib.Path(f"{SHIM}/gensim/__init__.py").write_text("")
pathlib.Path(f"{SHIM}/gensim/utils.py").write_text(
    "def tokenize(*a, **k):\n    raise NotImplementedError('gensim stub')\n")
pathlib.Path(f"{SHIM}/spacy/__init__.py").write_text(
    "__version__ = '0.0.0-stub'\n\n\n"
    "class Language:  # datasets.utils._dill checks issubclass(obj, spacy.Language)\n"
    "    pass\n\n\n"
    "def blank(*a, **k):\n    raise NotImplementedError('spacy stub')\n\n\n"
    "def load(*a, **k):\n    raise NotImplementedError('spacy stub')\n")
pathlib.Path(f"{SHIM}/spacy/symbols.py").write_text("ORTH = 65\n")

if SHIM not in sys.path:
    sys.path.insert(0, SHIM)

In [ ]:
import warnings

# Suppress DeprecationWarning
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=ResourceWarning)

Next, we import the required modules and check that a GPU is available.

In [ ]:
import gc
import glob
import io
import time
import urllib.request

import numpy as np
import pandas as pd
import torch
import lightning.pytorch as pl
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from torch.utils.data import DataLoader
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.random_projection import GaussianRandomProjection

from doubleml_deep import DoubleMLLightning
from doubleml_deep.plm import DoubleMLPLRDeep
from doubleml_deep.default_models import (
    DoubleMLDeepBeitModel,
    DoubleMLDeepRobertaModel,
    DoubleMLDeepSaintModel,
)

palette = sns.color_palette("colorblind")

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name} ({vram:.0f} GB)")
else:
    raise RuntimeError(
        "No GPU visible, so this notebook would train on CPU and take hours.\n"
        "Go to Runtime, Change runtime type, and pick a GPU (T4 is enough).\n"
        "TPU runtimes do not work here: torch sees no CUDA device and silently\n"
        "falls back to CPU. Supporting TPUs would need torch-xla.\n"
        "To override anyway, set ALLOW_CPU = True."
    )

We mount Google Drive at the beginning rather than at the end. During training, a checkpoint of about 3.5 GB is written to Drive after every epoch, so that a disconnected session can resume where it stopped. The final embeddings are also saved to Drive for use in Labs 3 to 5.

In [ ]:
DRIVE_DIR = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = "/content/drive/MyDrive/demand_labs_v2_ext"
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f"checkpoints and artifact go to {DRIVE_DIR}")
except Exception as exc:
    print(f"Drive not mounted ({type(exc).__name__}); checkpoints and artifact stay local "
          "and are lost when the session ends.")

## Constructing the Data Set

We construct the same data set as in Lab 1, using the same configuration and the same deterministic draw: all 7,549 products of the panel, split by product into 3,774 fine-tuning products ($I_1$) and 3,775 estimation products ($I_2$). The sample sizes are the paper's; the split is our own. If you change the configuration in Lab 1, change it here as well.

In [ ]:
# ---------------------------------------------------------------------------
# Configuration. Keep this cell identical in Labs 1 and 2.
# ---------------------------------------------------------------------------
BASE = ("https://raw.githubusercontent.com/JanTeichertKluge/"
        "demand-analysis-repro/main/main")
SEED = 42

# Products in the fine-tuning sample I_1; all other products form the
# estimation sample I_2. The sizes are the paper's, the draw is our own.
N_FINETUNE = 3774

# The panel comes in a train and a validation part. We load both and draw our
# own split by product.
PANEL_DIR = "data/amzn_toys_monthly_diffs_ffill_fixed_splits_buybox"
panel_files = [f"{PANEL_DIR}/{part}-0000{k}-of-00002.parquet"
               for part in ("train", "validation") for k in range(2)]

# (ASIN, date) keys of the observations the paper keeps once lags are formed.
# They decide which rows enter the analysis, not which split a product is in.
key_files = ["code/main_train_keys.csv", "code/main_val_keys.csv"]

# All 16 image part-files (about 1.3 GB). Lab 1 downloads only the first one.
IMAGE_DIR = "data/amzn_toys_monthly_avg_long_ffill_28_7_images"
image_files = [f"{IMAGE_DIR}/train-0000{i}-of-00004-part00{j}.parquet"
               for i in range(4) for j in range(4)]


def fetch(remote):
    """Download a file of the reproduction repository once; return the local path."""
    local = "data_cache/" + remote.split("/")[-1]
    if not os.path.exists(local):
        print("downloading", local, flush=True)
        urllib.request.urlretrieve(f"{BASE}/{remote}", local)
    return local


os.makedirs("data_cache", exist_ok=True)
for remote in panel_files + key_files:
    fetch(remote)

print("panel and sample keys ready (~42 MB, cached for this session)")

Next, we load the panel and construct the data set exactly as in Lab 1.

In [ ]:
columns = ["ASIN", "window", "date", "SALES_RANK", "BUYBOX_PRICE", "text",
           "RATING", "REVIEW_COUNT", "subcat_aggregated",
           "New Offer Count: Current",
           "Count of retrieved live offers: New, FBA",
           "Count of retrieved live offers: New, FBM",
           "Lightning Deals: Upcoming Deal", "Buy Box: Is FBA"]

panel = pd.concat([pd.read_parquet(fetch(f), columns=columns) for f in panel_files],
                  ignore_index=True)

# Keep the 28-day time averages, then every fourth week: 14 four-week periods
# from 2023-01-30 to 2024-01-29. SALES_RANK is already log(1 / sales rank) and
# BUYBOX_PRICE already log(price), so they are the paper's Q_it and P_it.
# As in the paper, the panel is unbalanced: a product enters with every period
# in which it is observed.
panel = panel[panel["window"] == 28].drop(columns="window").dropna()
dates = np.sort(panel["date"].unique())
panel["date_t"] = panel["date"].map({d: i for i, d in enumerate(dates)})
panel = panel[(panel["date_t"] <= 54) & (panel["date_t"] % 4 == 0)]
panel["period"] = panel["date_t"] // 4
panel = panel.rename(columns={"SALES_RANK": "Q_t", "BUYBOX_PRICE": "P_bb_t"})

df = panel.sort_values(["ASIN", "period"]).reset_index(drop=True)

# Deterministic split by product: every product, with all of its periods, is
# in exactly one sample. Labs 4 and 5 estimate only on the "estimation"
# products, so the embeddings they use were fitted on different products. The
# paper's Addendum A relies on that independence.
asins = np.sort(df["ASIN"].unique())
rng = np.random.default_rng(SEED)
finetune = set(rng.choice(asins, size=N_FINETUNE, replace=False))
df["split"] = np.where(df["ASIN"].isin(finetune), "finetune", "estimation")
assert df.groupby("ASIN")["split"].nunique().max() == 1, "a product is in both splits"

# Lagged state and first differences, within product. As in the paper, the lag
# is the product's previous observed period.
grouped = df.groupby("ASIN")
df["Q_t-1"] = grouped["Q_t"].shift(1)
df["P_bb_t-1"] = grouped["P_bb_t"].shift(1)
df["RATING_t-1"] = grouped["RATING"].shift(1)
df["REVIEW_COUNT_t-1"] = grouped["REVIEW_COUNT"].shift(1)
df["Delta_Q_t"] = df["Q_t"] - df["Q_t-1"]
df["Delta_P_bb_t"] = df["P_bb_t"] - df["P_bb_t-1"]

# Flag the rows the paper keeps: every row with a lagged state, minus a few
# that the paper's pipeline dropped. Labs 3 to 5 use only these rows.
keys = pd.concat([pd.read_csv(fetch(f), dtype=str) for f in key_files])
df["in_sample"] = pd.MultiIndex.from_frame(df[["ASIN", "date"]].astype(str)).isin(
    pd.MultiIndex.from_frame(keys[["ASIN", "date"]]))
assert df["in_sample"].sum() == len(keys)

print(df.groupby("split").agg(products=("ASIN", "nunique"), rows=("ASIN", "size"),
                              in_sample=("in_sample", "sum")))

## Training Settings

We use the settings of the paper's training script (`_src_scrpts/training_level_v02_bert_hpc.py`, the time-independent text-and-image run). The cell also fixes the session budget, the validation periods, and the dimensions used for the dimension reduction below.

In [ ]:
torch.set_float32_matmul_precision("medium")

BATCH_SIZE = 16
ACCUMULATE_GRAD_BATCHES = 2   # optimizer step every 2 batches: effective batch of 32
LEARNING_RATE = 2e-5
MOMENTUM = 0.95               # passed through as in the script; AdamW ignores it
WD = 1e-6
SCHEDULER = None              # constant learning rate
GRADIENT_CLIP_VAL = 2
DROPOUT = 0.1
FUSION_HEAD = [256, 128]

MAX_EPOCHS = 10               # upper bound; early stopping ends the run sooner
PATIENCE = 2                  # epochs without improvement before stopping
MIN_DELTA = 0.001             # smallest drop in val_combined_loss that counts

# Larger batches for extracting the embeddings, where no gradients are kept.
PREDICT_BATCH_SIZE = 64

# Colab: how long this session may train (the rest of the notebook needs roughly
# an hour more), and whether to continue a run an earlier session started.
TRAIN_HOURS = 3
RESUME = True

# Validation decides when training stops and which epoch's weights we keep:
# every estimation product observed in one of these two periods.
VAL_PERIODS = [0, 7]

N_CLUSTERS = 5          # k in k-means, as in the paper
N_COMPONENTS = 5        # number of PCA / similarity features
PROJECTION_DIM = 256    # Johnson-Lindenstrauss target dimension

torch.manual_seed(SEED)
pl.seed_everything(SEED, workers=True)

Next, we download all sixteen image part-files (about 1.3 GB) and collect the images of our products. Every product in the panel has an image. The source images are large: half of them are at least 1,500 pixels wide, and the largest are 2,560 × 2,560 pixels, which amounts to 19 MB once decoded. We therefore keep them encoded (about 115 KB each on average, 870 MB in total) and decode them one at a time during preprocessing below, so that the image processor sees exactly the images that the paper's pipeline saw. We read one part-file at a time to keep the memory use low.

In [ ]:
wanted = set(df["ASIN"])
images = {}
for f in image_files:
    table = pd.read_parquet(fetch(f))
    hit = table[table["ASIN"].isin(wanted)]
    images.update(zip(hit["ASIN"], (record["bytes"] for record in hit["image"])))
    del table, hit
    gc.collect()

missing = wanted - set(images)
assert not missing, f"{len(missing)} products have no image"
megabytes = sum(len(b) for b in images.values()) / 1e6
print(f"{len(images):,} product images ({megabytes:.0f} MB, still encoded)")

## Assembling the Inputs

The tabular input follows the paper's training script: the three offer counts (standardized), the lightning-deal indicator, the Buy Box FBA indicator and the subcategory dummies. Ratings, review counts, and lagged prices and quantities are not fed into the network; they enter later as ordinary controls in Labs 3 to 5.

The data are split by product. The network is trained on all periods of the fine-tuning products, so it sees each product 14 times with the same text and image but with different targets and tabular values. For validation, we use the estimation products in two periods. The validation loss determines when training stops and which epoch's weights we keep, just as the paper's script validated on held-out products. The network is never trained on these rows.

In [ ]:
# Fixed category list, so the tabular input width does not depend on which
# products happen to be drawn.
SUBCATS = sorted(panel["subcat_aggregated"].unique())
SUBCAT_COLS = [f"sub_{c.replace(' ', '_').replace('&', 'and')}" for c in SUBCATS]

dummies = (pd.get_dummies(df["subcat_aggregated"])
           .reindex(columns=SUBCATS, fill_value=0)
           .astype("float32"))
dummies.columns = SUBCAT_COLS
df = pd.concat([df, dummies], axis=1)

# Network copies of the time-varying tabular inputs; the artifact keeps the raw
# values. The script fits one scaler per split. We fit on the fine-tune rows and
# apply it to every row, so the estimation products do not shape the scaling.
COUNT_COLS = ["New Offer Count: Current",
              "Count of retrieved live offers: New, FBA",
              "Count of retrieved live offers: New, FBM"]
FLAG_COLS = ["Lightning Deals: Upcoming Deal", "Buy Box: Is FBA"]
X_COUNT_COLS = ["x_n_offers", "x_n_offers_fba", "x_n_offers_fbm"]
X_FLAG_COLS = ["x_lightning_deal", "x_is_fba"]

scaler = StandardScaler().fit(df.loc[df["split"] == "finetune", COUNT_COLS])
df[X_COUNT_COLS] = scaler.transform(df[COUNT_COLS])
df[X_FLAG_COLS] = df[FLAG_COLS].astype("float32").to_numpy()

TAB_NET_COLS = X_COUNT_COLS + X_FLAG_COLS + SUBCAT_COLS
print(f"{len(TAB_NET_COLS)} tabular columns ({len(SUBCAT_COLS)} subcategory dummies) "
      f"-> SAINT embedding of {32 * len(TAB_NET_COLS)} dimensions")

# Row positions in df: every period of the fine-tune products for training,
# every estimation product in VAL_PERIODS for validation.
is_finetune = (df["split"] == "finetune").to_numpy()
train_rows = np.flatnonzero(is_finetune)
val_rows = np.flatnonzero(~is_finetune & df["period"].isin(VAL_PERIODS).to_numpy())
print(f"training rows:   {len(train_rows):,} ({df.loc[train_rows, 'ASIN'].nunique()} products)")
print(f"validation rows: {len(val_rows):,} ({df.loc[val_rows, 'ASIN'].nunique()} products)")

The tabular input has 16 columns, 11 of which are subcategory dummies, and SAINT maps them into a 512-dimensional embedding. The network is trained on all 42,041 rows of the 3,774 fine-tuning products and validated on 5,247 rows of 2,995 estimation products, those observed in period 0 or 7.

## The Model

`DoubleMLPLRDeep` builds the network sketched above. Two settings deserve attention.

Setting `embedding_dim=None` makes the fusion layer the identity, so that the embedding we extract later is the raw layer-normalized concatenation $E_i \in \mathbb{R}^{2048}$ rather than a learned compression. The paper reduces the dimension afterwards by a random projection, which comes with distance-preservation guarantees. This setting also requires `use_attention=False`.

Setting `score="partialling out"` together with `combined_loss="multiplicative"` yields the loss $L = \|Q - \hat \ell\|\cdot\|P - \hat m\|$ defined above.

Building the model downloads roughly 600 MB of pre-trained weights. The resulting network has about 218 million trainable parameters.

In [ ]:
data_field_map = {
    "outcome": ["Q_t"],
    "treatments": ["P_bb_t"],
    "tab_cols": {"cat_cols": [], "con_cols": TAB_NET_COLS},
    "txt_cols": ["text"],
    "img_cols": ["image"],
}

model_dict = {
    # SAINT with the package defaults, as in the script. With no categorical
    # columns, cat_embed_input=None does what the script's [] does.
    "tab": DoubleMLDeepSaintModel(tab_cols=data_field_map["tab_cols"],
                                  cat_embed_input=None),
    "txt": DoubleMLDeepRobertaModel(padding=True),
    "img": DoubleMLDeepBeitModel(),
}

dml_deep = DoubleMLPLRDeep(
    data_field_map=data_field_map,
    model_dict=model_dict,
    fusion_head=FUSION_HEAD,
    embedding_dim=None,      # identity fusion -> raw concatenated embedding
    use_attention=False,     # required when embedding_dim is None
    score="partialling out",
    combined_loss="multiplicative",
    dropout_prob=DROPOUT,
)

lit_model = DoubleMLLightning(
    dml_deep, optim="AdamW", lr=LEARNING_RATE, weight_decay=WD, momentum=MOMENTUM,
    scheduler=SCHEDULER,
    save_hyperparameters=False,   # otherwise every checkpoint pickles the whole network
)

EMB_DIM = dml_deep.network.embedding_network.embedding_dim
n_train = sum(p.numel() for p in lit_model.parameters() if p.requires_grad)
print(f"embedding dimension: {EMB_DIM}")
print(f"trainable parameters: {n_train / 1e6:.1f}M")

## Data Loading

`doubleml-deep` also provides a data module, `DoubleMLDeepData`, which tokenizes and preprocesses every row up front and keeps the result in memory. With 83,777 rows, the preprocessed images alone would require far more memory than a Colab session has. Since descriptions and images do not change over time, we instead tokenize and preprocess each *product* once, using the same tokenizer and image processor, and assemble every batch from these caches. The network receives the same inputs that the data module would provide; only memory use and preprocessing time differ.

Even the per-product cache is large: 7,549 preprocessed images take about 4.5 GB in single precision. We store them in half precision (about 2.3 GB) and convert each batch back to single precision. Training and extraction run in 16-bit mixed precision, so the image encoder computes with half-precision inputs anyway.

As in the paper's script (`padding=True`), each batch pads its descriptions to the length of the longest description in the batch.

In [ ]:
tokenizer = model_dict["txt"].preprocessor
products = df.drop_duplicates("ASIN").set_index("ASIN")

t0 = time.time()
token_ids = {a: model_dict["txt"].preprocess([text])["input_ids"][0]
             for a, text in products["text"].items()}
pixel_values = {}
for a in products.index:
    with Image.open(io.BytesIO(images[a])) as im:   # one full-size image at a time
        pixel_values[a] = (model_dict["img"].preprocess(im.convert("RGB"))["pixel_values"][0]
                           .half())   # half precision halves the cache
gigabytes = sum(p.numel() * p.element_size() for p in pixel_values.values()) / 1e9
print(f"cached {len(token_ids):,} descriptions and {len(pixel_values):,} images "
      f"({gigabytes:.1f} GB) in {(time.time() - t0) / 60:.1f} min")

TARGET_TAB_COLS = ["Q_t", "P_bb_t"] + TAB_NET_COLS
row_values = torch.tensor(df[TARGET_TAB_COLS].to_numpy(dtype="float32"))
row_asins = df["ASIN"].to_numpy()


def collate(rows):
    """Assemble the network's input dict for a list of df row positions."""
    batch = dict(zip(TARGET_TAB_COLS, row_values[rows].T))
    ids = [token_ids[a] for a in row_asins[rows]]
    input_ids = torch.full((len(ids), max(map(len, ids))), tokenizer.pad_token_id)
    attention_mask = torch.zeros_like(input_ids)
    for i, t in enumerate(ids):   # pad to the longest description in the batch
        input_ids[i, :len(t)] = t
        attention_mask[i, :len(t)] = 1
    batch.update(input_ids=input_ids, attention_mask=attention_mask,
                 token_type_ids=torch.zeros_like(input_ids),
                 pixel_values=torch.stack([pixel_values[a] for a in row_asins[rows]]).float())
    return batch


def make_loader(rows, shuffle=False, drop_last=False, batch_size=BATCH_SIZE):
    return DataLoader(rows.tolist(), batch_size=batch_size, shuffle=shuffle,
                      drop_last=drop_last, collate_fn=collate,
                      generator=torch.Generator().manual_seed(SEED))


train_loader = make_loader(train_rows, shuffle=True, drop_last=True)
val_loader = make_loader(val_rows, drop_last=True)
print(f"{len(train_loader)} training batches per epoch, {len(val_loader)} validation batches")

## Training

We train with AdamW at a constant learning rate of $2 \cdot 10^{-5}$, weight decay $10^{-6}$, gradient clipping at norm 2, gradient accumulation over two batches and mixed precision. Training runs for at most 10 epochs and stops early once the validation loss has not improved by at least 0.001 for 2 consecutive epochs. Afterwards, we restore the weights of the best epoch. With a batch size of 16, one epoch consists of 2,627 training batches and 327 validation batches.

- *Session budget.* At the end of each epoch, the run checks whether another epoch still fits into `TRAIN_HOURS`. If not, it stops and the notebook continues with the best weights so far.
- *Resuming.* A full checkpoint is written to Drive after every epoch. Running the notebook again in a new session continues from this checkpoint, with the optimizer state, the epoch counter and the early-stopping state intact. Once early stopping or the epoch limit has ended the run, later runs skip training and load the best weights. Set `RESUME = False` to start over.

**Runtime.** On a T4 GPU, one epoch takes roughly half an hour, about four times as long as in the subsample version. With `TRAIN_HOURS = 3`, a session completes about five epochs, so a full run of 10 epochs takes two sessions unless early stopping ends it sooner. Adjust `TRAIN_HOURS` to the time your Colab plan allows.

At every validation epoch, the network also reports a partialling-out DML estimate $\hat\theta$ computed from its current predictions. The path of $\hat\theta$ across epochs is a useful diagnostic, but the estimate to rely on is the one in Lab 4, which is computed on held-out products with cross-fitting.

If the GPU runs out of memory, lower `BATCH_SIZE` to 8 and raise `ACCUMULATE_GRAD_BATCHES` to 4, which keeps the effective batch size at 32.

In [ ]:
RUN_DIR = os.path.join(DRIVE_DIR or ".", f"finetune_full_seed{SEED}")
RESUME_CKPT = os.path.join(RUN_DIR, "resume.ckpt")
DONE_FILE = os.path.join(RUN_DIR, "done.txt")
os.makedirs(RUN_DIR, exist_ok=True)
if not RESUME:
    for stale in (RESUME_CKPT, DONE_FILE):
        if os.path.exists(stale):
            os.remove(stale)


class SessionBudget(pl.Callback):
    """Print one line per epoch; stop at an epoch boundary before the budget runs out."""

    def __init__(self, hours):
        self.deadline = time.time() + 3600 * hours
        self.reached = False

    def on_train_start(self, trainer, pl_module):
        self.best = float(early_stopping.best_score)   # inf, or the restored best on resume

    def on_train_epoch_start(self, trainer, pl_module):
        self.epoch_start = time.time()

    def on_train_epoch_end(self, trainer, pl_module):
        now = time.time()
        loss = float(trainer.callback_metrics.get("val_combined_loss", float("nan")))
        self.best = min(self.best, loss)
        print(f"epoch {trainer.current_epoch:2d}: {(now - self.epoch_start) / 60:4.1f} min   "
              f"val_combined_loss {loss:.4f}   best {self.best:.4f}", flush=True)
        if not trainer.should_stop and now + 1.1 * (now - self.epoch_start) > self.deadline:
            self.reached = True
            trainer.should_stop = True


early_stopping = EarlyStopping(monitor="val_combined_loss", mode="min",
                               min_delta=MIN_DELTA, patience=PATIENCE)
best_ckpt = ModelCheckpoint(dirpath=RUN_DIR, filename="best-{epoch:03d}-{val_combined_loss:.4f}",
                            monitor="val_combined_loss", mode="min", save_top_k=1,
                            save_weights_only=True)
resume_ckpt = ModelCheckpoint(dirpath=RUN_DIR, filename="resume", save_top_k=1)  # full state, every epoch
budget = SessionBudget(TRAIN_HOURS)

trainer = pl.Trainer(
    accelerator="gpu",
    devices=1,
    precision="16-mixed",
    max_epochs=MAX_EPOCHS,
    gradient_clip_val=GRADIENT_CLIP_VAL,
    accumulate_grad_batches=ACCUMULATE_GRAD_BATCHES,
    callbacks=[early_stopping, best_ckpt, resume_ckpt, budget],
    logger=CSVLogger(save_dir=RUN_DIR, name="logs"),
    log_every_n_steps=20,
)

if "panel" in globals():
    del panel
gc.collect()

if RESUME and os.path.exists(DONE_FILE):
    with open(DONE_FILE) as fh:
        best_path = fh.read().strip()
    print("this run already finished in an earlier session; skipping training")
else:
    resume_from = RESUME_CKPT if RESUME and os.path.exists(RESUME_CKPT) else None
    if resume_from:
        print(f"resuming from {resume_from}")
    t0 = time.time()
    trainer.fit(lit_model, train_dataloaders=train_loader, val_dataloaders=val_loader,
                ckpt_path=resume_from)
    print(f"\nthis session trained for {(time.time() - t0) / 60:.0f} minutes")
    best_path = best_ckpt.best_model_path
    if budget.reached:
        print("stopped by the session budget. Run the notebook again in a new session to "
              "continue training; the steps below already use the best weights so far.")
    else:
        reason = "early stopping" if early_stopping.stopped_epoch > 0 else "epoch limit"
        print(f"training finished ({reason})")
        with open(DONE_FILE, "w") as fh:
            fh.write(best_path)

# Continue with the weights of the best validation epoch, not the last one.
best = torch.load(best_path, map_location="cpu", weights_only=False)
lit_model.load_state_dict(best["state_dict"])
print(f"restored the best epoch ({best['epoch']}): {os.path.basename(best_path)}")

The training log reports the validation loss after every epoch, and the notebook continues with the weights of the epoch with the lowest validation loss. Next, we plot the training and validation loss, the validation RMSE of the two prediction heads, and the in-training DML estimate.

In [ ]:
# one CSV per session; resumed sessions continue the step and epoch counters
logs = sorted(glob.glob(os.path.join(RUN_DIR, "logs", "version_*", "metrics.csv")))
metrics = pd.concat([pd.read_csv(p) for p in logs], ignore_index=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for col, lbl, ax in [("train_combined_loss", "train", axes[0]),
                     ("val_combined_loss", "validation", axes[0])]:
    if col in metrics:
        s = metrics[["step", col]].dropna()
        ax.plot(s["step"], s[col], label=lbl)
axes[0].set_title("Combined loss  $\\|Q-\\hat{l}\\|\\cdot\\|P-\\hat{m}\\|$")
axes[0].set_xlabel("step")
axes[0].legend()

for col, lbl in [("val_RMSE_ml_l_P_bb_t", "$\\|Q-\\hat{l}\\|$ (quantity)"),
                 ("val_RMSE_ml_m_P_bb_t", "$\\|P-\\hat{m}\\|$ (price)")]:
    if col in metrics:
        s = metrics[["epoch", col]].dropna()
        axes[1].plot(s["epoch"], s[col], marker="o", label=lbl)
axes[1].set_title("Validation RMSE by component")
axes[1].set_xlabel("epoch")
axes[1].legend()

if "theta_P_bb_t" in metrics:
    s = metrics[["epoch", "theta_P_bb_t"]].dropna()
    axes[2].plot(s["epoch"], s["theta_P_bb_t"], marker="o", color=palette[3])
    axes[2].axhline(-0.70, ls="--", color="gray", label="paper's estimate")
    axes[2].set_title("In-training DML estimate $\\hat{\\theta}$")
    axes[2].set_xlabel("epoch")
    axes[2].legend()

for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In the subsample version, the validation loss fell from 0.58 to about 0.50, the quantity head accounted for most of the loss, and the in-training estimate $\hat\theta$ settled between $-0.49$ and $-0.40$. With four times as many training products, expect a lower validation loss, in particular for the price head. The in-training estimate should still be smaller in magnitude than the paper's estimate of about $-0.7$ (dashed line), since the network conditions only on its own inputs and not on the lagged state, which Lab 4 identifies as the key confounder.

## Extracting the Embeddings

Since the tabular inputs change from period to period, we compute the embedding and the two nuisance predictions for every product-period, as in the paper's pipeline. This gives 83,777 rows. The batches are assembled from the same per-product caches as during training, but with `PREDICT_BATCH_SIZE` rows each.

The loader iterates over `df` in order (`shuffle=False`), so row $i$ of every output corresponds to row $i$ of `df`.

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

in_order = make_loader(np.arange(len(df)), batch_size=PREDICT_BATCH_SIZE)
(pred_Q, pred_P, E), _ = lit_model.predict_numpy(trainer, in_order)
df["pred_ml_l"] = pred_Q[:, 0]
df["pred_ml_m"] = pred_P[:, 0]

assert E.shape[0] == len(df), "embedding rows are not aligned with df"
print(f"embeddings: {E.shape}   (one row per product-period, {EMB_DIM} dimensions)")

print("\nR^2 of the nuisance predictions")
for split, g in df.groupby("split"):
    print(f"  {split:10s}  Q_t: {r2_score(g['Q_t'], g['pred_ml_l']):.3f}   "
          f"P_bb_t: {r2_score(g['P_bb_t'], g['pred_ml_m']):.3f}")

Expect the network's own predictions to fit the fine-tuning products better than the held-out estimation products. In the subsample version, the gap was large ($R^2$ of 0.84 versus 0.42 for $Q_{it}$), and with more training products it should shrink. Whatever its size, the gap illustrates why Labs 4 and 5 estimate only on the estimation products and re-estimate all nuisance functions with cross-fitting.

## Dimension Reduction

We follow Algorithm 1 of the paper's Online Appendix and apply it to every product-period row.

1. **Random projection.** We multiply the embeddings by a Gaussian random matrix to obtain $\bar{E}_i \in \mathbb{R}^{256}$. By the Johnson-Lindenstrauss lemma, this approximately preserves pairwise distances, so it reduces the dimension without distorting the geometry. Unlike a learned compression, it requires no fitting and cannot overfit.
2. **Centering and normalization.** We center the projected embeddings and normalize them to the unit sphere:
   $$X^e_i = \frac{\bar{E}_i - \frac{1}{n}\sum_j \bar{E}_j}
                  {\|\bar{E}_i - \frac{1}{n}\sum_j \bar{E}_j\|}.$$
   Products are then compared by direction rather than by magnitude.
3. **Summaries.** We compute the first five principal components,
$$
X^{pc}_{i,k} = \gamma_k^\top X^e_i,
$$
and the cosine similarities to the five $k$-means centroids, $X^{sim}_{i,k} = c_k^\top X^e_i$.

The paper works with the similarities rather than the principal components, because closeness to, say, a cluster of die-cast replicas is easier to interpret than a loading on the second principal axis. Lab 5 uses the similarities as the variables on which the elasticity depends.

In [ ]:
proj = GaussianRandomProjection(n_components=PROJECTION_DIM, random_state=SEED)
E_bar = proj.fit_transform(E)
X_e = normalize(E_bar - E_bar.mean(axis=0), axis=1)

pca = PCA(n_components=N_COMPONENTS, random_state=SEED)
X_pc = pca.fit_transform(X_e)

km = KMeans(n_clusters=N_CLUSTERS, random_state=SEED, n_init=200).fit(X_e)
X_sim = normalize(X_e) @ normalize(km.cluster_centers_).T

emb_cols = [f"emb_{i}" for i in range(PROJECTION_DIM)]
pca_cols = [f"pca_{i}" for i in range(N_COMPONENTS)]
sim_cols = [f"similarity_cluster_{i}" for i in range(N_CLUSTERS)]

# row i belongs to df.iloc[i]
row_features = pd.concat([
    pd.DataFrame(X_e, columns=emb_cols),
    pd.DataFrame(X_pc, columns=pca_cols),
    pd.DataFrame(X_sim, columns=sim_cols),
], axis=1)

print(f"X^e: {X_e.shape}   unit norm: "
      f"{np.allclose(np.linalg.norm(X_e, axis=1), 1.0)}")
print(f"variance explained by the first {N_COMPONENTS} PCs: "
      f"{pca.explained_variance_ratio_.sum():.1%}")
print("\nrows per cluster:")
print(pd.Series(km.labels_).value_counts().sort_index().to_string())

All rows of $X^e$ should have unit norm. The printout also reports the share of the variance of $X^e$ that the first five principal components explain and the number of product-period rows in each cluster.

## Inspecting the Embeddings

Before we use these vectors as controls, it is worth looking at them. We perform the two checks from the paper: we project the embeddings onto the first three principal components (for a random sample of 20,000 rows, to keep the plot legible) and color them by cluster, and we look at the products closest to each centroid.

In [ ]:
coords = PCA(n_components=3, random_state=SEED).fit_transform(X_e)
shown = np.random.default_rng(SEED).choice(len(X_e), size=min(20_000, len(X_e)),
                                           replace=False)

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
for k in range(N_CLUSTERS):
    m = shown[km.labels_[shown] == k]
    ax.scatter(coords[m, 0], coords[m, 1], coords[m, 2],
               s=3, alpha=0.3, color=palette[k], label=f"Cluster {k}")
ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
ax.set_zlabel("PC 3")
ax.set_title("Fine-tuned product embeddings, coloured by $k$-means cluster")
ax.legend(markerscale=3, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
n_show = 5
dists = km.transform(X_e)


def closest_products(k, n):
    """Rows of the n distinct products nearest to centroid k."""
    picked, seen = [], set()
    for idx in np.argsort(dists[:, k]):
        asin = df["ASIN"].iloc[idx]
        if asin not in seen:
            seen.add(asin)
            picked.append(idx)
            if len(picked) == n:
                break
    return picked


fig, axes = plt.subplots(N_CLUSTERS, n_show, figsize=(2.1 * n_show, 2.3 * N_CLUSTERS))
for k in range(N_CLUSTERS):
    for j, idx in enumerate(closest_products(k, n_show)):
        asin = df["ASIN"].iloc[idx]
        ax = axes[k, j]
        thumb = Image.open(io.BytesIO(images[asin]))
        thumb.thumbnail((224, 224))
        ax.imshow(thumb)
        ax.axis("off")
        if j == 0:
            ax.set_title(f"Cluster {k}", loc="left", fontsize=11, weight="bold")
plt.suptitle("Products closest to each cluster centroid", y=1.005)
plt.tight_layout()
plt.show()

for k in range(N_CLUSTERS):
    print(f"\n--- Cluster {k} ---")
    for idx in closest_products(k, 3):
        row = df.iloc[idx]
        print(f"  [{row['subcat_aggregated']}] {row['text'][:110]}")

Look for themes in the clusters. In the subsample version, the clusters were interpretable: licensed vehicles for young children, such as *Thomas & Friends* engines and *Paw Patrol* trucks; *Disney Pixar Cars* characters and die-cast replicas; carrier trucks for toddlers; larger playsets, track systems and remote-controlled vehicles; and die-cast models of luxury cars. The numbering of the clusters is arbitrary and changes from run to run.

## Saving the Embeddings

Labs 3, 4 and 5 run in separate Colab sessions, and the local `/content` directory does not persist across sessions. We therefore save the embeddings, together with the panel variables needed in the later labs, to Google Drive, which we mounted at the beginning.

If Drive is not mounted, the notebook only writes a local copy, which you can download and upload again in the next lab.

In [ ]:
TABULAR_RENAME = {
    "New Offer Count: Current": "n_offers",
    "Count of retrieved live offers: New, FBA": "n_offers_fba",
    "Count of retrieved live offers: New, FBM": "n_offers_fbm",
    "Lightning Deals: Upcoming Deal": "lightning_deal",
    "Buy Box: Is FBA": "is_fba",
}

keep = (["ASIN", "date", "period", "split", "subcat_aggregated",
         "Q_t", "P_bb_t", "Q_t-1", "P_bb_t-1", "Delta_Q_t", "Delta_P_bb_t",
         "RATING", "REVIEW_COUNT", "RATING_t-1", "REVIEW_COUNT_t-1",
         "in_sample", "pred_ml_l", "pred_ml_m"]
        + list(TABULAR_RENAME) + SUBCAT_COLS)

assert df.index.equals(row_features.index), "row features are not aligned with df"
artifact = pd.concat([df[keep].rename(columns=TABULAR_RENAME), row_features], axis=1)

assert artifact[emb_cols].notna().all().all(), "some rows lack an embedding"
print(f"artifact: {artifact.shape[0]:,} rows x {artifact.shape[1]:,} columns")
artifact.head(3)

In [ ]:
ARTIFACT_NAME = "full_v2_ext.parquet"
paths = [os.path.join("/content" if os.path.isdir("/content") else ".", ARTIFACT_NAME)]
if DRIVE_DIR:
    paths.insert(0, os.path.join(DRIVE_DIR, ARTIFACT_NAME))

for p in paths:
    artifact.to_parquet(p, index=False)
    print(f"wrote {p}  ({os.path.getsize(p) / 1e6:.1f} MB)")

In [ ]:
# Optional: download the artifact to your machine instead of using Drive.
# from google.colab import files
# files.download(paths[-1])

## Summary

We now have an embedding for every product-period that is fine-tuned to predict price and quantity, compressed to 256 dimensions and summarized by five principal components and five cluster similarities. The network was trained only on products that are not used for estimation in Labs 4 and 5. The saved data also contain the network's own nuisance predictions, `pred_ml_l` and `pred_ml_m`.

The clusters are interpretable, which indicates that the representation captures meaningful product characteristics. This alone does not show that the embeddings are useful for our purposes; for that, we need to know what they can predict.

* **Lab 3** measures how well the embeddings predict $Q$, $P$, $\Delta Q$ and $\Delta P$. The difference between levels and changes is central to the remaining labs.
* **Lab 4** uses the embeddings as controls to estimate the average price elasticity.
* **Lab 5** uses the cluster similarities as effect modifiers, allowing the elasticity to vary across products.